# Notebook 07a: Benchmark Calibration

**MSc Dissertation: Agentic AI for Sovereign Risk Assessment under Climate-Related Fiscal Stress**

This notebook calibrates the three conventional fiscal policy benchmarks that will serve as baselines
against which DQN and PPO agents are evaluated:

1. **Bohn (1998) Fiscal Reaction Function** — parameters estimated via OLS from the historical panel.
2. **Fixed Deficit Target (EU SGP)** — institutional rule parameters; no estimation needed.
3. **Passive Stress Test (NGFS-style)** — 2× climate stress multiplier applied to a do-nothing policy.

All validation runs use `SEED = 42`. Outputs are saved to `outputs/benchmarks/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
warnings.filterwarnings('ignore')

from src.environment.sovereign_risk_env import SovereignRiskEnv
from src.environment.config import load_profile, list_profiles
from src.benchmarks.bohn_fiscal_reaction import FiscalPolicy, RandomPolicy, BohnFiscalReaction
from src.benchmarks.fixed_deficit_target import FixedDeficitTarget
from src.benchmarks.passive_stress_test import PassiveStressTest

# Paths
CONFIG_PATH   = '../data/processed/transition_parameters.json'
SCALING_PATH  = '../data/processed/scaling_parameters.json'
PANEL_PATH    = '../data/processed/master_panel_engineered.csv'
BENCH_DIR     = '../outputs/benchmarks'
FIGURES_DIR   = '../outputs/figures'
os.makedirs(BENCH_DIR,   exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

SEED     = 42
PROFILES = list_profiles(CONFIG_PATH)

def make_env(profile, **kwargs):
    return SovereignRiskEnv(
        profile=profile, config_path=CONFIG_PATH,
        scaling_path=SCALING_PATH, **kwargs
    )

print(f'Profiles ({len(PROFILES)}): {PROFILES}')
print('Setup complete.')

---
## Shared Evaluation Helper

All three benchmarks are evaluated using the same `evaluate_policy()` function,
ensuring results are comparable.

In [ ]:
def evaluate_policy(policy, profile, n_episodes=100, seed=42, max_steps=30, **env_kwargs):
    """Run n_episodes and return summary statistics.
    
    Returns a dict with:
        mean_terminal_debt, median_terminal_debt, mean_cumulative_reward,
        pct_crisis, action_distribution (dict mapping action→fraction),
        all_terminal_debts (list, for box plots)
    """
    env = make_env(profile, max_steps=max_steps, **env_kwargs)
    
    terminal_debts   = []
    cumul_rewards    = []
    crisis_count     = 0
    action_counts    = {i: 0 for i in range(6)}
    total_steps      = 0
    
    policy.reset()
    
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        policy.reset()
        ep_reward = 0.0
        
        for _ in range(max_steps):
            action = policy.select_action(obs, info)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            action_counts[action] += 1
            total_steps += 1
            if terminated or truncated:
                if terminated:
                    crisis_count += 1
                break
        
        terminal_debts.append(info['raw_state']['debt'])
        cumul_rewards.append(ep_reward)
    
    env.close()
    
    action_dist = {k: v / max(total_steps, 1) for k, v in action_counts.items()}
    
    return {
        'mean_terminal_debt':   np.mean(terminal_debts),
        'median_terminal_debt': np.median(terminal_debts),
        'std_terminal_debt':    np.std(terminal_debts),
        'mean_cumulative_reward': np.mean(cumul_rewards),
        'pct_crisis':           100.0 * crisis_count / n_episodes,
        'action_distribution':  action_dist,
        'all_terminal_debts':   terminal_debts,
    }

ACTION_NAMES = {
    0: 'Severe austerity',
    1: 'Moderate austerity',
    2: 'Maintain',
    3: 'Moderate stimulus',
    4: 'Large stimulus',
    5: 'Climate adaptation',
}

print('evaluate_policy() helper defined.')

---
## Part 1: Calibrate the Bohn Fiscal Reaction Function

### Economic background

Bohn (1998) showed that a positive fiscal response to rising debt — i.e., the primary balance
improves when debt is high — is a *sufficient condition for intertemporal solvency*. The model is:

$$\text{pb}(t) = \alpha + \beta \cdot d(t-1) + \gamma \cdot \text{output\_gap}(t) + \varepsilon(t)$$

where $\beta > 0$ (Bohn's sustainability condition) means the government tightens when debt rises.

We estimate this via pooled OLS within each economy type (Advanced / Emerging Market / Developing)
over 2000–2024 non-projection observations.

In [ ]:
# ── 1a. Load and clean the panel ──────────────────────────────────────────
df = pd.read_csv(PANEL_PATH)
print(f'Full panel: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Years: {df["year"].min():.0f} – {df["year"].max():.0f}')
print(f'Countries: {df["iso3"].nunique()}')

# Filter to: 2000–2024 actuals, reasonable fiscal values
reg_df = df[
    (df['year'] >= 2000) &
    (df['year'] <= 2024) &
    (df['is_projection'] == False) &
    (df['weo_primary_balance'].between(-50, 50)) &   # remove clear data errors
    (df['debt_to_gdp_lag1'].between(0, 300)) &        # remove extreme outliers
    df['output_gap'].notna()
].copy()

reg_vars = ['economy_type', 'iso3', 'year', 'weo_primary_balance',
            'debt_to_gdp_lag1', 'output_gap']
reg_df = reg_df[reg_vars].dropna()

print(f'\nRegression sample: {len(reg_df):,} observations')
print('By economy type:')
print(reg_df.groupby('economy_type').size().to_string())
print('\nDescriptive statistics — primary balance (% GDP):')
print(reg_df.groupby('economy_type')['weo_primary_balance'].describe().round(3).to_string())

In [ ]:
# ── 1b. OLS regression by economy type ────────────────────────────────────
# We estimate pooled OLS (no fixed effects) as a proof-of-concept consistent
# with the original Bohn (1998) specification. For a production system,
# country fixed effects (linearmodels.PanelOLS) would be preferred to absorb
# country-specific structural primary balances.

print('=' * 70)
print('BOHN (1998) FISCAL REACTION FUNCTION — OLS ESTIMATES')
print('Dependent variable: weo_primary_balance (% GDP)')
print('Period: 2000–2024, non-projection observations')
print('Sample filter: |pb| ≤ 50%, debt_lag1 ∈ [0%, 300%]')
print('=' * 70)

coefficients = {}
ols_models   = {}

for econ_type in ['Advanced', 'Emerging Market', 'Developing']:
    sub = reg_df[reg_df['economy_type'] == econ_type].copy()
    X = sm.add_constant(sub[['debt_to_gdp_lag1', 'output_gap']])
    y = sub['weo_primary_balance']
    
    model = sm.OLS(y, X).fit(cov_type='HC3')  # heteroskedasticity-robust SEs
    ols_models[econ_type] = model
    
    alpha = model.params['const']
    beta  = model.params['debt_to_gdp_lag1']
    gamma = model.params['output_gap']
    
    se_beta  = model.bse['debt_to_gdp_lag1']
    se_gamma = model.bse['output_gap']
    pv_beta  = model.pvalues['debt_to_gdp_lag1']
    pv_gamma = model.pvalues['output_gap']
    
    sustainability = 'YES ✓' if beta > 0 else 'NO  ✗'
    
    coefficients[econ_type] = {
        'alpha': round(alpha, 4),
        'beta':  round(beta,  4),
        'gamma': round(gamma, 4),
        'r_squared': round(model.rsquared, 4),
        'n_obs': int(model.nobs),
        'se_beta': round(se_beta, 4),
        'se_gamma': round(se_gamma, 4),
        'pv_beta': round(pv_beta, 4),
        'pv_gamma': round(pv_gamma, 4),
    }
    
    print(f'\n--- {econ_type} (N={int(model.nobs):,}) ---')
    print(f'  α (intercept)    = {alpha:+.4f}')
    print(f'  β (debt_lag1)    = {beta:+.4f}  (SE={se_beta:.4f}, p={pv_beta:.4f})')
    print(f'  γ (output gap)   = {gamma:+.4f}  (SE={se_gamma:.4f}, p={pv_gamma:.4f})')
    print(f'  R²               = {model.rsquared:.4f}')
    print(f'  Bohn β > 0?      → {sustainability}')
    
    if beta > 0:
        sig = '***' if pv_beta < 0.01 else ('**' if pv_beta < 0.05 else ('*' if pv_beta < 0.10 else 'n.s.'))
        print(f'  Sustainability:  β is positive {sig} — Bohn condition satisfied.')
    else:
        print(f'  WARNING: β < 0 — primary balance deteriorates as debt rises.')
        print(f'  This is empirically common for developing economies with limited fiscal space.')

print('\n' + '=' * 70)
print('DISSERTATION NOTE: The Bohn fiscal reaction function is estimated via')
print('pooled OLS within each economy type over 2000–2024. A positive β')
print('indicates governments systematically adjust primary balances in response')
print('to rising debt, satisfying Bohn\'s (1998) intertemporal solvency condition.')
print('='*70)

In [ ]:
# ── 1c. Visualise regression fit ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colours = {'Advanced': '#1f77b4', 'Emerging Market': '#ff7f0e', 'Developing': '#2ca02c'}

for ax, econ_type in zip(axes, ['Advanced', 'Emerging Market', 'Developing']):
    sub = reg_df[reg_df['economy_type'] == econ_type]
    c   = coefficients[econ_type]
    
    ax.scatter(sub['debt_to_gdp_lag1'], sub['weo_primary_balance'],
               alpha=0.25, s=12, color=colours[econ_type], label='Data')
    
    d_range = np.linspace(sub['debt_to_gdp_lag1'].min(), sub['debt_to_gdp_lag1'].max(), 100)
    pb_fit  = c['alpha'] + c['beta'] * d_range
    ax.plot(d_range, pb_fit, 'k-', linewidth=2, label=f"β={c['beta']:+.4f}")
    
    ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Lagged Debt-to-GDP (%)', fontsize=11)
    ax.set_ylabel('Primary Balance (% GDP)', fontsize=11)
    ax.set_title(f'{econ_type}\nN={c["n_obs"]:,}, R²={c["r_squared"]:.4f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

plt.suptitle('Bohn (1998) Fiscal Reaction Function — OLS Fit by Economy Type',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_bohn_regression.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_bohn_regression')

In [ ]:
# ── 1d. Build CalibratedBohn instances for all 9 profiles ─────────────────
from src.benchmarks.calibrated_bohn import CalibratedBohn, BOHN_COEFFICIENTS

# Update the module-level BOHN_COEFFICIENTS with the freshly estimated values
import src.benchmarks.calibrated_bohn as _cb_module
_cb_module.BOHN_COEFFICIENTS.update(coefficients)

calibrated_bohn = {}
for profile in PROFILES:
    cfg       = load_profile(profile, CONFIG_PATH, SCALING_PATH)
    econ_type = cfg.economy_type
    c         = coefficients[econ_type]
    
    calibrated_bohn[profile] = CalibratedBohn(
        alpha=c['alpha'],
        beta=c['beta'],
        gamma=c['gamma'],
        debt_target=60.0,
        growth_base=cfg.growth_base,
        economy_type=econ_type,
        profile_name=profile,
    )

print('CalibratedBohn instances created for all profiles:')
for p, b in calibrated_bohn.items():
    print(f'  {p:<30} {b}')

In [ ]:
# ── 1e. Validate Bohn benchmarks on the 3 Medium profiles ─────────────────
print('Running Bohn validation (100 episodes each, SEED=42)...')
print()

MEDIUM_PROFILES = ['Advanced_Medium', 'Emerging_Market_Medium', 'Developing_Medium']
bohn_validation = {}

for profile in MEDIUM_PROFILES:
    res = evaluate_policy(calibrated_bohn[profile], profile, n_episodes=100, seed=SEED)
    bohn_validation[profile] = res
    
    print(f'{profile}:')
    print(f'  Mean terminal debt  : {res["mean_terminal_debt"]:>8.1f}% GDP')
    print(f'  Median terminal debt: {res["median_terminal_debt"]:>8.1f}% GDP')
    print(f'  Mean cumul. reward  : {res["mean_cumulative_reward"]:>8.1f}')
    print(f'  Crisis rate         : {res["pct_crisis"]:>7.1f}%')
    print(f'  Action distribution :')
    for a, frac in res['action_distribution'].items():
        if frac > 0.01:
            print(f'    Action {a} ({ACTION_NAMES[a]:<22}): {100*frac:5.1f}%')
    print()

---
## Part 2: Verify the Fixed Deficit Target Benchmark (EU SGP)

### Economic background

The EU Stability and Growth Pact (SGP) imposes two numerical fiscal rules:
1. **Deficit criterion**: overall deficit must not exceed 3% of GDP.
2. **Debt criterion**: debt must be below 60% of GDP, or declining sufficiently toward it.

No calibration is needed — these *are* the institutional rules. We only need to verify the benchmark
produces economically sensible behaviour: more austerity for high-debt profiles, less for low-debt ones.

In [ ]:
# ── 2a. Print SGP parameters ───────────────────────────────────────────────
sgp = FixedDeficitTarget(deficit_target=-3.0, debt_target=60.0, adjustment_speed=0.5)

print('=' * 60)
print('FIXED DEFICIT TARGET (EU STABILITY AND GROWTH PACT)')
print('=' * 60)
print(f'  deficit_target    = {sgp.deficit_target:.1f}% GDP  (EU Maastricht criterion)')
print(f'  debt_target       = {sgp.debt_target:.1f}% GDP  (EU Maastricht criterion)')
print(f'  adjustment_speed  = {sgp.adjustment_speed:.1f}   (close 50% of gap per year)')
print()
print('DISSERTATION NOTE: The fixed deficit target benchmark implements a')
print('simplified version of the EU SGP, targeting a maximum structural')
print('deficit of 3% GDP and a debt target of 60%, with a 50% annual')
print('adjustment speed toward the target (European Commission, 2020).')

In [ ]:
# ── 2b. Validate SGP benchmark on 3 Medium profiles ───────────────────────
print('Running SGP validation (100 episodes each, SEED=42)...')
print()

sgp_validation = {}

for profile in MEDIUM_PROFILES:
    res = evaluate_policy(sgp, profile, n_episodes=100, seed=SEED)
    sgp_validation[profile] = res
    
    print(f'{profile}:')
    print(f'  Mean terminal debt  : {res["mean_terminal_debt"]:>8.1f}% GDP')
    print(f'  Median terminal debt: {res["median_terminal_debt"]:>8.1f}% GDP')
    print(f'  Mean cumul. reward  : {res["mean_cumulative_reward"]:>8.1f}')
    print(f'  Crisis rate         : {res["pct_crisis"]:>7.1f}%')
    print(f'  Action distribution :')
    for a, frac in res['action_distribution'].items():
        if frac > 0.01:
            print(f'    Action {a} ({ACTION_NAMES[a]:<22}): {100*frac:5.1f}%')
    print()

---
## Part 3: Passive Stress Test — NGFS-Style Climate Stress Scenario

### Economic background

The Bank of England's Climate Biennial Exploratory Scenario (CBES) and NGFS scenarios assess climate
risk by applying a predefined physical stress to institutions that do *not* adapt. In fiscal terms:
the government maintains its current policy while facing heightened climate shocks.

The **NGFS Hot House World** scenario projects physical risk approximately **doubling by mid-century**
under a >3°C warming pathway. We implement this as a 2× multiplier on:
- `climate_event_probability` (more frequent disasters)
- `climate_conditional_mean` (larger average damage when events occur)

This table shows the **fiscal cost of climate inaction** for each of the 9 profiles.

In [ ]:
# ── 3a. Define stress evaluation function ─────────────────────────────────
def evaluate_stress(profile, n_episodes=100, seed=42, stress_multiplier=2.0):
    """Run passive stress test under both baseline and stressed climate conditions.
    
    The environment is instantiated normally; for stressed runs, the climate
    parameters on env.config are modified in-place after creation (before
    episodes begin). This is safe because reset() re-reads from env.config
    at the start of every episode, and the dynamics functions read from config.
    """
    policy = PassiveStressTest()
    
    # Baseline (normal climate)
    normal = evaluate_policy(policy, profile, n_episodes=n_episodes, seed=seed)
    
    # Stressed: modify config climate parameters after construction
    env_stress = make_env(profile)
    orig_prob  = env_stress.config.climate_event_probability
    orig_mean  = env_stress.config.climate_conditional_mean
    
    env_stress.config.climate_event_probability = min(orig_prob * stress_multiplier, 1.0)
    env_stress.config.climate_conditional_mean  = orig_mean * stress_multiplier
    
    # Run stressed episodes
    terminal_debts  = []
    cumul_rewards   = []
    crisis_count    = 0
    
    for ep in range(n_episodes):
        obs, info = env_stress.reset(seed=seed + ep)
        ep_reward = 0.0
        for _ in range(30):
            action = policy.select_action(obs, info)
            obs, reward, terminated, truncated, info = env_stress.step(action)
            ep_reward += reward
            if terminated or truncated:
                if terminated:
                    crisis_count += 1
                break
        terminal_debts.append(info['raw_state']['debt'])
        cumul_rewards.append(ep_reward)
    
    env_stress.close()
    
    stressed = {
        'mean_terminal_debt':     np.mean(terminal_debts),
        'median_terminal_debt':   np.median(terminal_debts),
        'mean_cumulative_reward': np.mean(cumul_rewards),
        'pct_crisis':             100.0 * crisis_count / n_episodes,
        'all_terminal_debts':     terminal_debts,
        'stress_prob':            orig_prob * stress_multiplier,
        'stress_mean':            orig_mean * stress_multiplier,
    }
    
    return normal, stressed

print('evaluate_stress() function defined.')

In [ ]:
# ── 3b. Run stress test across all 9 profiles ──────────────────────────────
STRESS_MULTIPLIER = 2.0
print(f'Running passive stress test (NGFS Hot House World, {STRESS_MULTIPLIER}× climate risk)...')
print(f'Using {100} episodes per profile × 2 scenarios (normal + stressed)')
print()

stress_results = {}
for profile in PROFILES:
    normal, stressed = evaluate_stress(profile, n_episodes=100, seed=SEED,
                                        stress_multiplier=STRESS_MULTIPLIER)
    stress_results[profile] = {'normal': normal, 'stressed': stressed}

# Print comparison table
print(f'{'Profile':<28} | {'Normal':^34} | {'Stressed (2×)':^34} | {'Δ Debt':>8}')
print(f'{'':28} | {'Mean Debt':^12} {'Crisis%':^10} {'Reward':^10} | '
      f'{'Mean Debt':^12} {'Crisis%':^10} {'Reward':^10} |')
print('-' * 100)

for profile in PROFILES:
    n = stress_results[profile]['normal']
    s = stress_results[profile]['stressed']
    delta = s['mean_terminal_debt'] - n['mean_terminal_debt']
    print(
        f"{profile:<28} | "
        f"{n['mean_terminal_debt']:>10.1f}%  {n['pct_crisis']:>7.1f}%  {n['mean_cumulative_reward']:>8.1f}  | "
        f"{s['mean_terminal_debt']:>10.1f}%  {s['pct_crisis']:>7.1f}%  {s['mean_cumulative_reward']:>8.1f}  | "
        f"{delta:>+8.1f}pp"
    )

print()
print('DISSERTATION NOTE: The passive stress test follows the NGFS Hot House')
print('World scenario, applying a 2× multiplier to climate event probability')
print('and conditional damage, consistent with projected physical risk increases')
print('under a >3°C warming pathway (NGFS, 2023).')

In [ ]:
# ── 3c. Visualise stress test impact ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

x         = np.arange(len(PROFILES))
width     = 0.35
norm_debts    = [stress_results[p]['normal']['mean_terminal_debt']   for p in PROFILES]
stressed_debts = [stress_results[p]['stressed']['mean_terminal_debt'] for p in PROFILES]

bars1 = ax.bar(x - width/2, norm_debts,    width, label='Baseline climate', color='#5B9BD5', alpha=0.85)
bars2 = ax.bar(x + width/2, stressed_debts, width, label='Stressed (2×)',    color='#d62728', alpha=0.85)

ax.axhline(60,  color='green', linestyle='--', linewidth=1.3, alpha=0.7, label='Maastricht 60%')
ax.axhline(200, color='red',   linestyle='--', linewidth=1.3, alpha=0.7, label='Crisis threshold 200%')

short_labels = [p.replace('_', '\n') for p in PROFILES]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=8.5)
ax.set_ylabel('Mean Terminal Debt-to-GDP (%)', fontsize=11)
ax.set_title(f'Fiscal Cost of Climate Inaction: Passive Policy, Baseline vs. NGFS Stressed ({STRESS_MULTIPLIER}×)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, axis='y', alpha=0.25)

# Vertical separators for economy groups
for xv in [2.5, 5.5]:
    ax.axvline(xv, color='grey', linestyle=':', alpha=0.5)

plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_stress_test_comparison.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_stress_test_comparison')

---
## Part 4: Full Benchmark Comparison — All 9 Profiles

In [ ]:
# ── 4a. Run all three benchmarks on all 9 profiles ─────────────────────────
print('Running full benchmark comparison (100 episodes × 9 profiles × 3 benchmarks)...')
print('(This will take ~2 minutes)')
print()

bench_results = {}

for profile in PROFILES:
    bench_results[profile] = {}
    
    # Bohn
    bench_results[profile]['bohn'] = evaluate_policy(
        calibrated_bohn[profile], profile, n_episodes=100, seed=SEED)
    
    # SGP
    bench_results[profile]['sgp'] = evaluate_policy(
        sgp, profile, n_episodes=100, seed=SEED)
    
    # Passive (normal)
    bench_results[profile]['passive'] = stress_results[profile]['normal']
    
    # Random (for reference)
    bench_results[profile]['random'] = evaluate_policy(
        RandomPolicy(), profile, n_episodes=100, seed=SEED)
    
    print(f'  {profile:<30}: done')

print('\nAll benchmarks complete.')

In [ ]:
# ── 4b. Summary table ─────────────────────────────────────────────────────
print('=' * 100)
print(f'  BENCHMARK COMPARISON: Mean Terminal Debt (% GDP) | Crisis Rate (%)  |  Mean Reward')
print(f'  {'Profile':<28}  {'Bohn':^22} {'SGP':^22} {'Passive':^22} {'Random':^22}')
print('=' * 100)

for profile in PROFILES:
    row = bench_results[profile]
    def fmt(key):
        r = row[key]
        return f"{r['mean_terminal_debt']:6.1f}% | {r['pct_crisis']:4.1f}% | {r['mean_cumulative_reward']:6.0f}"
    print(f"  {profile:<28}  {fmt('bohn')}   {fmt('sgp')}   {fmt('passive')}   {fmt('random')}")

print('=' * 100)

In [ ]:
# ── 4c. Box plot comparison across all profiles ────────────────────────────
profile_order = [
    'Advanced_Low', 'Advanced_Medium', 'Advanced_High',
    'Emerging_Market_Low', 'Emerging_Market_Medium', 'Emerging_Market_High',
    'Developing_Low', 'Developing_Medium', 'Developing_High',
]

bench_labels = ['Bohn', 'SGP', 'Passive', 'Random']
bench_keys   = ['bohn', 'sgp', 'passive', 'random']
bench_colours = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharey=True)
axes_flat = axes.flatten()

for idx, profile in enumerate(profile_order):
    ax = axes_flat[idx]
    data = [bench_results[profile][k]['all_terminal_debts'] for k in bench_keys]
    
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color='black', linewidth=1.8),
                    flierprops=dict(marker='o', markersize=2.5, alpha=0.4),
                    whiskerprops=dict(linewidth=1.3),
                    capprops=dict(linewidth=1.3))
    
    for patch, col in zip(bp['boxes'], bench_colours):
        patch.set_facecolor(col)
        patch.set_alpha(0.6)
    
    ax.axhline(60,  color='green', linestyle='--', linewidth=1,   alpha=0.6)
    ax.axhline(200, color='red',   linestyle='--', linewidth=1,   alpha=0.6)
    ax.set_xticks([1,2,3,4])
    ax.set_xticklabels(bench_labels, fontsize=8)
    ax.set_title(profile.replace('_', ' '), fontsize=9, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.25)

# Common y-label
for ax in axes[:, 0]:
    ax.set_ylabel('Terminal Debt (% GDP)', fontsize=9)

# Legend
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, alpha=0.6, label=l)
                  for c, l in zip(bench_colours, bench_labels)]
fig.legend(handles=legend_patches, fontsize=10, ncol=4,
           loc='lower center', bbox_to_anchor=(0.5, -0.01))

fig.suptitle('Benchmark Terminal Debt Distributions — All 9 Profiles (100 Episodes Each)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_benchmark_comparison.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_benchmark_comparison')

---
## Part 5: Save Calibrated Configurations

In [ ]:
# ── 5a. Assemble and save benchmark_config.json ───────────────────────────

# Collect validation results summary
validation_summary = {}
for profile in PROFILES:
    r = bench_results[profile]
    validation_summary[profile] = {
        bk: {
            'mean_terminal_debt':     round(r[bk]['mean_terminal_debt'],   2),
            'median_terminal_debt':   round(r[bk]['median_terminal_debt'], 2),
            'mean_cumulative_reward': round(r[bk]['mean_cumulative_reward'], 2),
            'pct_crisis':             round(r[bk]['pct_crisis'], 2),
        }
        for bk in bench_keys
    }

# Stress test results
stress_summary = {}
for profile in PROFILES:
    n = stress_results[profile]['normal']
    s = stress_results[profile]['stressed']
    stress_summary[profile] = {
        'normal':  {'mean_terminal_debt': round(n['mean_terminal_debt'], 2),
                    'pct_crisis':          round(n['pct_crisis'], 2),
                    'mean_cumulative_reward': round(n['mean_cumulative_reward'], 2)},
        'stressed': {'mean_terminal_debt': round(s['mean_terminal_debt'], 2),
                     'pct_crisis':          round(s['pct_crisis'], 2),
                     'mean_cumulative_reward': round(s['mean_cumulative_reward'], 2),
                     'stress_prob':          round(s['stress_prob'], 4),
                     'stress_mean':          round(s['stress_mean'], 4)},
        'delta_mean_debt': round(s['mean_terminal_debt'] - n['mean_terminal_debt'], 2),
    }

# Bohn coefficients (only the numeric keys for the JSON)
bohn_coeffs_clean = {
    et: {k: v for k, v in c.items() if k not in ('se_beta','se_gamma','pv_beta','pv_gamma')}
    for et, c in coefficients.items()
}

benchmark_config = {
    "calibration_note": "Notebook 07a — OLS on 2000–2024 WEO/World Bank panel, filtered |pb|<=50%, debt_lag1 in [0,300%]",
    "seed": SEED,
    "bohn": {
        "coefficients": bohn_coeffs_clean,
        "debt_target":  60.0,
        "methodology":  "Pooled OLS by economy type, 2000-2024, non-projection observations",
    },
    "sgp": {
        "deficit_target":   -3.0,
        "debt_target":       60.0,
        "adjustment_speed": 0.5,
        "reference":        "European Commission (2020) Vade Mecum SGP, IP-129",
    },
    "passive_stress": {
        "stress_multiplier": STRESS_MULTIPLIER,
        "parameters_stressed": ["climate_event_probability", "climate_conditional_mean"],
        "rationale": "NGFS Hot House World: 2× physical risk by mid-century under >3°C warming (NGFS 2023)",
    },
    "validation_results": validation_summary,
    "stress_test_results": stress_summary,
}

config_path = f'{BENCH_DIR}/benchmark_config.json'
with open(config_path, 'w') as f:
    json.dump(benchmark_config, f, indent=2)

print(f'Saved: {config_path}')
print(f'File size: {os.path.getsize(config_path):,} bytes')

In [ ]:
# ── 5b. Final summary ─────────────────────────────────────────────────────
print('=' * 70)
print('=== BENCHMARK CALIBRATION COMPLETE ===')
print('=' * 70)
print()
print('Bohn (1998) OLS Coefficients:')
for et, c in coefficients.items():
    sust = 'β>0 ✓' if c['beta'] > 0 else 'β<0 ✗'
    print(f'  {et:<18}: α={c["alpha"]:+.4f}  β={c["beta"]:+.4f}  γ={c["gamma"]:+.4f}  R²={c["r_squared"]:.4f}  N={c["n_obs"]:,}  [{sust}]')

print()
print(f'SGP: deficit_target=-3.0%, debt_target=60%, adjustment_speed=0.5')
print(f'Passive stress multiplier: {STRESS_MULTIPLIER}×')

print()
print('Validation — Mean terminal debt (100 episodes, Medium profiles):')
for profile in MEDIUM_PROFILES:
    r = bench_results[profile]
    print(f'  {profile:<30}: Bohn={r["bohn"]["mean_terminal_debt"]:6.1f}%  '
          f'SGP={r["sgp"]["mean_terminal_debt"]:6.1f}%  '
          f'Passive={r["passive"]["mean_terminal_debt"]:6.1f}%  '
          f'Random={r["random"]["mean_terminal_debt"]:6.1f}%')

print()
print('Stress test — Δ mean terminal debt (stressed − baseline):')
for profile in PROFILES:
    delta = stress_summary[profile]['delta_mean_debt']
    bar = '▓' * min(int(abs(delta)/5), 20)
    print(f'  {profile:<30}: {delta:+7.1f}pp  {bar}')

print()
print(f'Saved: outputs/benchmarks/benchmark_config.json')
print(f'Saved: src/benchmarks/calibrated_bohn.py')
print(f'Figures: outputs/figures/fig_bohn_regression.*')
print(f'         outputs/figures/fig_stress_test_comparison.*')
print(f'         outputs/figures/fig_benchmark_comparison.*')
print('=' * 70)